# Jupiter Held-Suarez / Lian-Showman relaxation model

A Newtonian-relaxation GCM for Jupiter, following the Held-Suarez test-case
structure but with a Jupiter-like equilibrium temperature profile from
Lian & Showman (2008, 2010), running on the `dinosaur` JAX dynamical core.
This runs on GPU if one is available (`jax.devices()` below).

In [ ]:
import sys
sys.path.insert(0, '.')  # so `jupiter_gcm_utils` (in this directory) imports

import dinosaur
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import jupiter_gcm_utils as jgu

units = dinosaur.scales.units

print('jax version:', jax.__version__)
print('devices:', jax.devices())

## Grid, physical constants, and forcing

`dinosaur.scales` on this branch is set to Jupiter's physical constants (radius, rotation rate, gravity, heat capacity, kappa), so `PrimitiveEquationsSpecs.from_si()` gives Jupiter values by default. The vertical grid uses `equidistant_log` sigma levels (concentrated near the top) so the deep, high-pressure Jovian atmosphere is adequately resolved with a manageable number of layers.

In [ ]:
layers = 60
scale_heights = 11
p0 = 25e5 * units.pascal  # deep reference pressure

coords = jgu.build_coords(layers=layers, scale_heights=scale_heights, horizontal_grid='T42')
physics_specs = dinosaur.primitive_equations.PrimitiveEquationsSpecs.from_si()

state, ref_temps, orography = jgu.initial_state(coords, physics_specs, p0=p0)

hs_forcing = dinosaur.held_suarez.LianShowmanForcing(
    coords=coords,
    physics_specs=physics_specs,
    reference_temperature=ref_temps,
    p0=p0,
)

primitive = dinosaur.primitive_equations.PrimitiveEquations(
    ref_temps, orography, coords, physics_specs
)

## Diagnostic forcing fields

Sanity-check the relaxation forcing itself before running the dynamical core: the Rayleigh-drag coefficient `kv`, the Newtonian-cooling coefficient `kt`, and the equilibrium temperature `Teq` it relaxes towards.

In [ ]:
grid = coords.horizontal
sigma = coords.vertical.centers
lon, _ = grid.nodal_mesh
surface_pressure_nondim = physics_specs.nondimensionalize(p0) * np.ones_like(lon)

lat_deg = np.arcsin(grid.nodal_axes[1]) * 180 / np.pi

kv_si = physics_specs.dimensionalize(hs_forcing.kv()[:, 0, 0], 1 / units.day)
kt_si = physics_specs.dimensionalize(hs_forcing.kt(), 1 / units.day)
teq_si = physics_specs.dimensionalize(
    hs_forcing.equilibrium_temperature(surface_pressure_nondim), units.degK
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(kv_si.magnitude, sigma)
axes[0].set_ylim(1, 0)
axes[0].set_xlabel('kv (1/day)')
axes[0].set_ylabel('sigma')
axes[0].set_title('Rayleigh drag coefficient')

cf = axes[1].contourf(lat_deg, sigma, kt_si.magnitude[:, 0, :], levels=30)
axes[1].set_ylim(1, 0)
axes[1].set_xlabel('latitude')
axes[1].set_title('kt (1/day) at lon=0')
plt.colorbar(cf, ax=axes[1])

cf = axes[2].contourf(lat_deg, sigma, teq_si.magnitude[:, 0, :], levels=30)
axes[2].set_ylim(1, 0)
axes[2].set_xlabel('latitude')
axes[2].set_title('Teq (K) at lon=0')
plt.colorbar(cf, ax=axes[2])
plt.tight_layout()

In [ ]:
# Radiative-equilibrium potential temperature at the equator, log-pressure axis,
# to check the deep-isothermal / cubic-taper / constant-potential-temperature
# structure of the Lian-Showman profile looks as expected.
pressure_si = (sigma[:, None, None] * p0.magnitude) * units.pascal
kappa = dinosaur.scales.KAPPA.magnitude
potential_temperature = teq_si * (pressure_si / p0) ** (-kappa)

eq_idx = np.argmin(np.abs(lat_deg))
plt.figure(figsize=(5, 5))
plt.plot(potential_temperature.magnitude[:, 0, eq_idx], sigma)
plt.yscale('log')
plt.gca().invert_yaxis()
plt.xlabel('Equilibrium potential temperature (K)')
plt.ylabel('sigma')
plt.title('Equatorial radiative-equilibrium profile')

In [ ]:
final_state, ds, elapsed = jgu.run_integration(
    [primitive, hs_forcing],
    coords,
    physics_specs,
    state,
    ref_temps,
    dt_si=10 * units.minute,
    save_every=1 * units.day,
    total_time=5 * units.day,
)
ds

In [ ]:
n_steps = int((5 * units.day) / (10 * units.minute))
print(f'{elapsed:.1f}s for {n_steps} steps on {jax.devices()[0].platform.upper()}'
      f' ({elapsed / n_steps * 1000:.1f} ms/step)')

final = ds.isel(time=-1)
u_zonal_mean = final.u.mean('lon')
t_zonal_mean = final.temperature.mean('lon')

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

cf = axes[0].contourf(final.lat, final.sigma, u_zonal_mean, levels=30, cmap='RdBu_r')
axes[0].set_ylim(1, 0)
axes[0].set_xlabel('latitude')
axes[0].set_ylabel('sigma')
axes[0].set_title('Zonal-mean zonal wind (m/s)')
plt.colorbar(cf, ax=axes[0])

cf = axes[1].contourf(final.lat, final.sigma, t_zonal_mean, levels=30)
axes[1].set_ylim(1, 0)
axes[1].set_xlabel('latitude')
axes[1].set_title('Zonal-mean temperature (K)')
plt.colorbar(cf, ax=axes[1])
plt.tight_layout()

plt.figure(figsize=(5, 4))
ds.total_kinetic_energy.plot()
plt.title('Total kinetic energy vs time')

## Post-integration diagnostics

Zonal-mean zonal wind and temperature at the final saved time, plus kinetic energy over the run (as a basic sanity check that nothing has blown up) and a wall-clock timing readout.

## Short GPU integration

Compose the primitive-equation dynamical core with the Lian-Showman forcing and integrate forward. This is deliberately short (a few days) as a smoke test that the model runs correctly and quickly on the GPU; increase `total_time` for a real spin-up (order ~1000s of days for Jupiter's weak thermal relaxation to matter).